# 2.0-feature-engineering

## Imports

In [ ]:
import pandas as pd
from src.config.config import (
    AIR_AREA_COL,
    AIR_GENRE_COL,
    AIR_RESTAURANT_ID_COL,
    CALENDAR_DATE_COL,
    HPG_RESTAURANT_ID_COL,
    LATITUDE_COL,
    LONGITUDE_COL,
    VISIT_DATE_COL,
    VISITORS_COL,
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    RAW_DATA_DIR,
)
from src.config.features import (
    CITY_COL,
    DAY_OF_WEEK_COL,
    DAYS_OF_WEEK,
    OPEN_DATE_COL,
    RESERVE_AIR_COL,
    RESERVE_AIR_NBR_COL,
    RESERVE_HPG_COL,
    RESERVE_HPG_NBR_COL,
    TOTAL_RES_COL,
    TOTAL_RES_NBR_COL,
    VISITORS_NBR_COL,
    VISITORS_DOW_MEAN_COL,
    VISITORS_DOW_MEAN_NBR_COL,
    RES_VISITORS_DIFF_COL,
    RES_VISITORS_DIFF_NBR_COL,
    GENRE_TE,
    AREA_TE,
)
from src.dataset import (
    prepare_datetime_columns,
    standardize_date,
)
from src.features import (
    add_basic_stats,
    add_days_since_last_record,
    add_golden_week_flg,
    add_historical_dow_mean,
    add_holiday_columns,
    add_lags,
    add_last_month_visitors,
    add_nbrs_reserves,
    add_neighbors_stats,
    add_open_usually,
    add_opened_recently_flg,
    add_reserves_difference,
    add_sum_of_reserves,
    add_time_based_target_encoding,
    add_total_nbr_reserves,
    add_total_reserves,
    drop_first_month,
    get_open_by_weekday_pct,
    get_open_status,
)
from src.plots import plot_corr_matrix

In [3]:
air_visit_df = pd.read_csv(INTERIM_DATA_DIR / 'air_visit.csv')
air_reserve_df = pd.read_csv(INTERIM_DATA_DIR / 'air_reserve.csv')
hpg_reserve_df = pd.read_csv(INTERIM_DATA_DIR / 'hpg_reserve.csv')
future_df = pd.read_csv(INTERIM_DATA_DIR / 'sample_submission.csv')
air_store_df = pd.read_csv(INTERIM_DATA_DIR / 'air_store_info.csv')
hpg_store_df = pd.read_csv(INTERIM_DATA_DIR / 'hpg_store_info.csv')
date_info_df = pd.read_csv(RAW_DATA_DIR / 'date_info.csv')
store_rel_df = pd.read_csv(RAW_DATA_DIR / 'store_id_relation.csv')

C:\Users\lymuthien\AppData\Local\Temp\ipykernel_14024\3750369184.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  hpg_reserve_df = pd.read_csv(INTERIM_DATA_DIR / 'hpg_reserve.csv')


In [4]:
prepare_datetime_columns(air_reserve_df)
prepare_datetime_columns(hpg_reserve_df)

standardize_date(date_info_df, CALENDAR_DATE_COL)
standardize_date(air_visit_df, VISIT_DATE_COL)
standardize_date(future_df, VISIT_DATE_COL)

## Features

### Air & hpg stores

#### Opening days

A one-hot encoding for the days of the week is needed. This data can be extracted based on the percentage of gaps (after processing, 0 visitors per day). In this case, there should most likely be no reservations for that day.

In [5]:
pct_df = get_open_by_weekday_pct(air_visit_df, VISIT_DATE_COL, VISITORS_COL)
air_store_df = air_store_df.merge(
    get_open_status(pct_df),
    on=AIR_RESTAURANT_ID_COL,
    how='left'
)
air_store_df = air_store_df.drop([AIR_AREA_COL, LATITUDE_COL, LONGITUDE_COL], axis=1)
air_store_df.head()

,air_store_id,air_genre_name,city,Mon_open,Tue_open,Wed_open,Th_open,Fri_open,Sat_open,Sun_open
0,air_0f0cdeee6c9bf3d7,Italian/French,Hyōgo-ken,1,1,1,1,1,1,1
1,air_7cc17a324ae5c7dc,Italian/French,Hyōgo-ken,0,1,1,1,1,1,1
2,air_fee8dcf4d619598e,Italian/French,Hyōgo-ken,1,1,1,1,1,1,1
3,air_a17f0778617c76e2,Italian/French,Hyōgo-ken,1,1,1,1,1,1,1
4,air_83db5aff8f50478e,Italian/French,Tōkyō-to,1,1,1,1,1,1,1


### Reserve dataframes

For further work, it is necessary to know the total number of reservations in the restaurant per day.

In [6]:
air_res_sum = add_sum_of_reserves(air_reserve_df, RESERVE_AIR_COL)
air_res_sum = air_res_sum.merge(
    air_store_df[[AIR_RESTAURANT_ID_COL, CITY_COL]],
    on=AIR_RESTAURANT_ID_COL
)
air_res_sum = add_nbrs_reserves(air_res_sum, RESERVE_AIR_COL, CITY_COL, RESERVE_AIR_NBR_COL)
air_res_sum.head()

,air_store_id,visit_date,air_reserves,city,air_reserves_nbrs
0,air_00a91d42b08b08d9,2016-10-31,2,Tōkyō-to,7.533333
1,air_00a91d42b08b08d9,2016-12-05,9,Tōkyō-to,8.116279
2,air_00a91d42b08b08d9,2016-12-14,18,Tōkyō-to,12.362069
3,air_00a91d42b08b08d9,2016-12-17,2,Tōkyō-to,15.342857
4,air_00a91d42b08b08d9,2016-12-20,4,Tōkyō-to,14.333333


In [7]:
air_res_sum[RESERVE_AIR_NBR_COL].isna().sum()

np.int64(0)

However, air_reserve dataframe still has a large number of gaps.

In [8]:
hpg_res_sum = add_sum_of_reserves(hpg_reserve_df, RESERVE_HPG_COL, HPG_RESTAURANT_ID_COL)
hpg_res_sum = hpg_res_sum.merge(
    hpg_store_df[[HPG_RESTAURANT_ID_COL, CITY_COL]],
    on=HPG_RESTAURANT_ID_COL
)
hpg_res_sum = add_nbrs_reserves(hpg_res_sum, RESERVE_HPG_COL, CITY_COL, RESERVE_HPG_NBR_COL)
hpg_res_sum.head()

,hpg_store_id,visit_date,hpg_reserves,city,hpg_reserves_nbrs
0,hpg_001ce40a1f873e4f,2016-01-13,4,Hyōgo-ken,5.861111
1,hpg_001ce40a1f873e4f,2016-01-27,7,Hyōgo-ken,6.191489
2,hpg_001ce40a1f873e4f,2016-02-13,2,Hyōgo-ken,5.925926
3,hpg_001ce40a1f873e4f,2016-02-27,8,Hyōgo-ken,8.017544
4,hpg_001ce40a1f873e4f,2016-03-16,2,Hyōgo-ken,6.491525


In [9]:
hpg_res_sum[RESERVE_HPG_NBR_COL].isna().sum()

np.int64(0)

In [10]:
hpg_res_sum_mapped = hpg_res_sum.merge(
    store_rel_df,
    on=HPG_RESTAURANT_ID_COL
)

### Date info

It is necessary to add a feature for the distance to the nearest holiday.

In [11]:
date_info_df = add_holiday_columns(date_info_df, CALENDAR_DATE_COL)

It is also necessary to designate Golden Week, since not all days of this week are holidays.

In [12]:
date_info_df = add_golden_week_flg(date_info_df, [2016, 2017], CALENDAR_DATE_COL)

Later we will need to match the weekday of a selected date with the restaurant’s open/closed flag for that weekday, so it makes sense to rename the days.

In [13]:
FULL_WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
mapping = dict(zip(FULL_WEEKDAYS, DAYS_OF_WEEK))
date_info_df[DAY_OF_WEEK_COL] = date_info_df[DAY_OF_WEEK_COL].replace(mapping)

### Air visit

#### Start dates

We can consider the restaurant's opening date. Since simply counting the number of days since opening may not be sufficient due to the increase in days over time, it's best to flag the restaurant's opening as occurring within the last six months. A **potential issue**: some restaurants either planned to open earlier than their minimum opening date but didn't, or didn't report visitors for earlier dates. This is indicated by the fact that air_reserve dataframe has reservations for earlier dates.

In [14]:
air_open_dates = (
    air_visit_df.groupby(AIR_RESTAURANT_ID_COL)[VISIT_DATE_COL]
    .min()
    .rename(OPEN_DATE_COL)
)

In [15]:
air_visit_df = add_opened_recently_flg(
    air_visit_df, air_open_dates, VISIT_DATE_COL, AIR_RESTAURANT_ID_COL
)
future_df = add_opened_recently_flg(
    future_df, air_open_dates, VISIT_DATE_COL, AIR_RESTAURANT_ID_COL
)

In [16]:
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1


In [17]:
future_df.head()

,id,visitors,air_store_id,visit_date,open_date,opened_recently
0,air_00a91d42b08b08d9_2017-04-23,0,air_00a91d42b08b08d9,2017-04-23,2016-07-01,0
1,air_00a91d42b08b08d9_2017-04-24,0,air_00a91d42b08b08d9,2017-04-24,2016-07-01,0
2,air_00a91d42b08b08d9_2017-04-25,0,air_00a91d42b08b08d9,2017-04-25,2016-07-01,0
3,air_00a91d42b08b08d9_2017-04-26,0,air_00a91d42b08b08d9,2017-04-26,2016-07-01,0
4,air_00a91d42b08b08d9_2017-04-27,0,air_00a91d42b08b08d9,2017-04-27,2016-07-01,0


#### Open dates

In [18]:
air_visit_df = air_visit_df.merge(
    date_info_df,
    left_on=VISIT_DATE_COL,
    right_on=CALENDAR_DATE_COL
).merge(
    air_store_df,
    on=AIR_RESTAURANT_ID_COL
).drop(CALENDAR_DATE_COL, axis=1)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,city,Mon_open,Tue_open,Wed_open,Th_open,Fri_open,Sat_open,Sun_open
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1,Fri,0,-17,0,Italian/French,Tōkyō-to,1,1,1,1,1,1,0
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1,Sat,0,-16,0,Italian/French,Tōkyō-to,1,1,1,1,1,1,0
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1,Sun,0,-15,0,Italian/French,Tōkyō-to,1,1,1,1,1,1,0
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1,Mon,0,-14,0,Italian/French,Tōkyō-to,1,1,1,1,1,1,0
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1,Tue,0,-13,0,Italian/French,Tōkyō-to,1,1,1,1,1,1,0


It is necessary to match the day_of_week column with the opening columns depending on the day of the week.

In [19]:
air_visit_df = add_open_usually(air_visit_df)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,city,open_usually
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1,Fri,0,-17,0,Italian/French,Tōkyō-to,1
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1,Sat,0,-16,0,Italian/French,Tōkyō-to,1
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1,Sun,0,-15,0,Italian/French,Tōkyō-to,0
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1,Mon,0,-14,0,Italian/French,Tōkyō-to,1
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1,Tue,0,-13,0,Italian/French,Tōkyō-to,1


In [20]:
future_df = future_df.merge(
    date_info_df,
    left_on=VISIT_DATE_COL,
    right_on=CALENDAR_DATE_COL
).merge(
    air_store_df,
    on=AIR_RESTAURANT_ID_COL
).drop(CALENDAR_DATE_COL, axis=1)
future_df = add_open_usually(future_df)
future_df.head()

,id,visitors,air_store_id,visit_date,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,city,open_usually
0,air_00a91d42b08b08d9_2017-04-23,0,air_00a91d42b08b08d9,2017-04-23,2016-07-01,0,Sun,0,-6,0,Italian/French,Tōkyō-to,0
1,air_00a91d42b08b08d9_2017-04-24,0,air_00a91d42b08b08d9,2017-04-24,2016-07-01,0,Mon,0,-5,0,Italian/French,Tōkyō-to,1
2,air_00a91d42b08b08d9_2017-04-25,0,air_00a91d42b08b08d9,2017-04-25,2016-07-01,0,Tue,0,-4,0,Italian/French,Tōkyō-to,1
3,air_00a91d42b08b08d9_2017-04-26,0,air_00a91d42b08b08d9,2017-04-26,2016-07-01,0,Wed,0,-3,0,Italian/French,Tōkyō-to,1
4,air_00a91d42b08b08d9_2017-04-27,0,air_00a91d42b08b08d9,2017-04-27,2016-07-01,0,Th,0,-2,0,Italian/French,Tōkyō-to,1


#### Days since the last visit record

Let's say if a restaurant was open on the previous day and the current day, the value will be zero. Otherwise, it will be the number of days since the last opening plus one.

However, test dataframe has a problem: it doesn't have a concept of gaps in dates, meaning it's impossible to definitively determine when restaurants were open or closed. We must either rely on further calculation of the opening hours or remove this feature altogether.

In [21]:
air_visit_df = add_days_since_last_record(air_visit_df, AIR_RESTAURANT_ID_COL, VISIT_DATE_COL)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,city,open_usually,days_from_last_visit
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1,Fri,0,-17,0,Italian/French,Tōkyō-to,1,0
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1,Sat,0,-16,0,Italian/French,Tōkyō-to,1,0
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1,Sun,0,-15,0,Italian/French,Tōkyō-to,0,1
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1,Mon,0,-14,0,Italian/French,Tōkyō-to,1,2
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1,Tue,0,-13,0,Italian/French,Tōkyō-to,1,0


In [22]:
future_df = add_days_since_last_record(future_df, AIR_RESTAURANT_ID_COL, VISIT_DATE_COL, air_visit_df)
future_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,city,open_usually,days_from_last_visit,id
0,air_00a91d42b08b08d9,2017-04-23,0,2016-07-01,0,Sun,0,-6,0,Italian/French,Tōkyō-to,0,1,air_00a91d42b08b08d9_2017-04-23
1,air_00a91d42b08b08d9,2017-04-24,0,2016-07-01,0,Mon,0,-5,0,Italian/French,Tōkyō-to,1,2,air_00a91d42b08b08d9_2017-04-24
2,air_00a91d42b08b08d9,2017-04-25,0,2016-07-01,0,Tue,0,-4,0,Italian/French,Tōkyō-to,1,0,air_00a91d42b08b08d9_2017-04-25
3,air_00a91d42b08b08d9,2017-04-26,0,2016-07-01,0,Wed,0,-3,0,Italian/French,Tōkyō-to,1,0,air_00a91d42b08b08d9_2017-04-26
4,air_00a91d42b08b08d9,2017-04-27,0,2016-07-01,0,Th,0,-2,0,Italian/French,Tōkyō-to,1,0,air_00a91d42b08b08d9_2017-04-27


#### Mean visitors by air city.

It is necessary to make smoothed target encoding, while the average value will be used for new areas. In this case, it is worth considering only days when the number of visitors is not zero, so that only open restaurants are taken into account.

In [23]:
air_visit_df, future_df = add_time_based_target_encoding(
    air_visit_df,
    future_df,
    CITY_COL,
    VISITORS_COL,
    AREA_TE
)

In [24]:
air_visit_df[
    [AIR_RESTAURANT_ID_COL, VISITORS_COL, AREA_TE, CITY_COL, VISIT_DATE_COL]
].head()

,air_store_id,visitors,air_city_te,city,visit_date
0,air_00a91d42b08b08d9,35,19.900494,Tōkyō-to,2016-07-01
1,air_00a91d42b08b08d9,9,19.979643,Tōkyō-to,2016-07-02
2,air_00a91d42b08b08d9,0,20.035936,Tōkyō-to,2016-07-03
3,air_00a91d42b08b08d9,20,20.042395,Tōkyō-to,2016-07-04
4,air_00a91d42b08b08d9,25,19.987284,Tōkyō-to,2016-07-05


In [25]:
future_df[[AIR_RESTAURANT_ID_COL, AREA_TE, CITY_COL, VISIT_DATE_COL]].head()

,air_store_id,air_city_te,city,visit_date
0,air_00a91d42b08b08d9,18.734807,Tōkyō-to,2017-04-23
1,air_00a91d42b08b08d9,18.734807,Tōkyō-to,2017-04-24
2,air_00a91d42b08b08d9,18.734807,Tōkyō-to,2017-04-25
3,air_00a91d42b08b08d9,18.734807,Tōkyō-to,2017-04-26
4,air_00a91d42b08b08d9,18.734807,Tōkyō-to,2017-04-27


#### Mean visitors by air genre

It is necessary to make smoothed target encoding, while the average value will be used for new genres.

In [26]:
air_visit_df, future_df = add_time_based_target_encoding(
    air_visit_df,
    future_df,
    AIR_GENRE_COL,
    VISITORS_COL,
    GENRE_TE
)

In [27]:
air_visit_df[
    [AIR_RESTAURANT_ID_COL, VISITORS_COL, GENRE_TE, AIR_GENRE_COL, VISIT_DATE_COL]
].head()

,air_store_id,visitors,air_genre_te,air_genre_name,visit_date
0,air_00a91d42b08b08d9,35,20.901991,Italian/French,2016-07-01
1,air_00a91d42b08b08d9,9,20.966002,Italian/French,2016-07-02
2,air_00a91d42b08b08d9,0,21.038925,Italian/French,2016-07-03
3,air_00a91d42b08b08d9,20,21.059025,Italian/French,2016-07-04
4,air_00a91d42b08b08d9,25,21.017889,Italian/French,2016-07-05


In [28]:
future_df[[AIR_RESTAURANT_ID_COL, GENRE_TE, AIR_GENRE_COL, VISIT_DATE_COL]].head()

,air_store_id,air_genre_te,air_genre_name,visit_date
0,air_00a91d42b08b08d9,20.680881,Italian/French,2017-04-23
1,air_00a91d42b08b08d9,20.680881,Italian/French,2017-04-24
2,air_00a91d42b08b08d9,20.680881,Italian/French,2017-04-25
3,air_00a91d42b08b08d9,20.680881,Italian/French,2017-04-26
4,air_00a91d42b08b08d9,20.680881,Italian/French,2017-04-27


#### Total reserved visitors

Total reserved visitors (from air_reserve and hpg_reserve) on this day - for this restaurant/for neighbors.

By neighbors, we designate restaurants that are located in the same city.

In [29]:
air_visit_df = add_total_reserves(air_visit_df, air_res_sum, hpg_res_sum_mapped, CITY_COL)
air_visit_df = add_total_nbr_reserves(air_visit_df, hpg_res_sum, CITY_COL)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,city,open_usually,days_from_last_visit,air_city_te,air_genre_te,total_reservations,total_reservations_nbrs
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1,Fri,0,-17,0,Italian/French,Tōkyō-to,1,0,19.900494,20.901991,0.0,8.448052
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1,Sat,0,-16,0,Italian/French,Tōkyō-to,1,0,19.979643,20.966002,0.0,8.133874
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1,Sun,0,-15,0,Italian/French,Tōkyō-to,0,1,20.035936,21.038925,0.0,6.724265
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1,Mon,0,-14,0,Italian/French,Tōkyō-to,1,2,20.042395,21.059025,0.0,6.730964
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1,Tue,0,-13,0,Italian/French,Tōkyō-to,1,0,19.987284,21.017889,0.0,7.102941


In [30]:
future_df = add_total_reserves(future_df, air_res_sum, hpg_res_sum_mapped, CITY_COL)
future_df = add_total_nbr_reserves(future_df, hpg_res_sum, CITY_COL)
future_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,city,open_usually,days_from_last_visit,id,air_city_te,air_genre_te,total_reservations,total_reservations_nbrs
0,air_00a91d42b08b08d9,2017-04-23,0,2016-07-01,0,Sun,0,-6,0,Italian/French,Tōkyō-to,0,1,air_00a91d42b08b08d9_2017-04-23,18.734807,20.680881,0.0,7.443182
1,air_00a91d42b08b08d9,2017-04-24,0,2016-07-01,0,Mon,0,-5,0,Italian/French,Tōkyō-to,1,2,air_00a91d42b08b08d9_2017-04-24,18.734807,20.680881,0.0,10.388350
2,air_00a91d42b08b08d9,2017-04-25,0,2016-07-01,0,Tue,0,-4,0,Italian/French,Tōkyō-to,1,0,air_00a91d42b08b08d9_2017-04-25,18.734807,20.680881,0.0,8.569038
3,air_00a91d42b08b08d9,2017-04-26,0,2016-07-01,0,Wed,0,-3,0,Italian/French,Tōkyō-to,1,0,air_00a91d42b08b08d9_2017-04-26,18.734807,20.680881,0.0,11.735354
4,air_00a91d42b08b08d9,2017-04-27,0,2016-07-01,0,Th,0,-2,0,Italian/French,Tōkyō-to,1,0,air_00a91d42b08b08d9_2017-04-27,18.734807,20.680881,0.0,9.649886


#### Rolling mean/median/std of visitors

If open_usually == 0 and visitors == 0, the value is replaced with NaN. Otherwise, zero is included in the aggregation. Later, we need to take care of the initial rows of the window, because right now they contain non-NaN values, even though conceptually they should be NaN.

In [31]:
aggs = [
    ("mean", {}),
    ("median", {}),
    ("std", {"ddof": 0}),
    ("max", {}),
    ("min", {}),
]
air_visit_df = add_basic_stats(air_visit_df, VISITORS_COL, AIR_RESTAURANT_ID_COL, aggs=aggs)
air_visit_df = add_neighbors_stats(air_visit_df, VISITORS_COL, CITY_COL)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,...,visitors_nbrs,visitors_nbrs_mean_7,visitors_nbrs_median_7,visitors_nbrs_std_7,visitors_nbrs_mean_14,visitors_nbrs_median_14,visitors_nbrs_std_14,visitors_nbrs_mean_28,visitors_nbrs_median_28,visitors_nbrs_std_28
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1,Fri,0,-17,0,Italian/French,...,24.935252,20.675402,19.949045,2.734276,20.384452,19.587426,3.120575,19.911380,18.720498,3.229918
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1,Sat,0,-16,0,Italian/French,...,23.669100,20.886135,19.949045,2.986030,20.399109,19.587426,3.141363,20.019059,18.720498,3.343359
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1,Sun,0,-15,0,Italian/French,...,20.609121,20.831720,19.949045,2.930840,20.328889,19.587426,3.055318,20.058484,18.720498,3.379464
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1,Mon,0,-14,0,Italian/French,...,15.793296,20.430933,19.949045,2.735545,20.204331,19.587426,3.005401,20.014863,18.720498,3.364158
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1,Tue,0,-13,0,Italian/French,...,17.550122,20.282356,19.949045,2.946981,20.199020,19.587426,3.013115,20.037469,18.720498,3.333447


#### Visitors on the same day last month


The feature has a drawback: if the previous month had 30 days and the current month has 31, then the feature for the 31st day will correspond to the 30th day. If there were 0 visitors, and the open_usually flag is 0, then 0 visitors are indicated.

In [32]:
air_visit_df = add_last_month_visitors(air_visit_df, VISITORS_COL)
air_visit_df.tail()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,...,visitors_nbrs_mean_7,visitors_nbrs_median_7,visitors_nbrs_std_7,visitors_nbrs_mean_14,visitors_nbrs_median_14,visitors_nbrs_std_14,visitors_nbrs_mean_28,visitors_nbrs_median_28,visitors_nbrs_std_28,visitors_last_month
296883,air_fff68b929994bfbd,2017-04-18,6,2016-07-01,0,Tue,0,-11,0,Bar/Cocktail,...,20.026217,19.502326,4.170304,19.816555,19.598581,3.434192,20.151737,19.966650,3.265143,8.0
296884,air_fff68b929994bfbd,2017-04-19,2,2016-07-01,0,Wed,0,-10,0,Bar/Cocktail,...,20.715592,19.502326,3.561120,19.927295,19.598581,3.392685,20.340102,19.966650,3.055621,2.0
296885,air_fff68b929994bfbd,2017-04-20,2,2016-07-01,0,Th,0,-9,0,Bar/Cocktail,...,20.884069,19.593458,3.475110,19.920054,19.547892,3.393281,20.355848,19.966650,3.050593,2.0
296886,air_fff68b929994bfbd,2017-04-21,4,2016-07-01,0,Fri,0,-8,0,Bar/Cocktail,...,20.769351,19.593458,3.531625,19.957683,19.547892,3.376573,20.309810,19.867153,3.065482,2.0
296887,air_fff68b929994bfbd,2017-04-22,5,2016-07-01,0,Sat,0,-7,0,Bar/Cocktail,...,20.796238,19.593458,3.553487,20.059399,19.547892,3.466848,20.303655,19.867153,3.058372,1.0


#### Visitors lag features

In [33]:
lags = [1, 7, 28]
air_visit_df = add_lags(
    air_visit_df,
    AIR_RESTAURANT_ID_COL,
    VISITORS_COL,
    lags
)
air_visit_df = add_lags(
    air_visit_df,
    CITY_COL,
    VISITORS_NBR_COL,
    lags,
    True
)
air_visit_df.tail()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,...,visitors_nbrs_mean_28,visitors_nbrs_median_28,visitors_nbrs_std_28,visitors_last_month,visitors_lag_1,visitors_lag_7,visitors_lag_28,visitors_nbrs_lag_1,visitors_nbrs_lag_7,visitors_nbrs_lag_28
254021,air_db4b38ebe7a7ceff,2017-04-22,19,2016-01-01,0,Sat,0,-7,0,Dining bar,...,19.778918,19.499812,3.881836,1.0,17.0,28.0,25.0,22.243243,25.121622,27.783784
255091,air_dc0e080ba0a5e5af,2017-04-22,10,2016-07-01,0,Sat,0,-7,0,Dining bar,...,19.778918,19.499812,3.881836,6.0,10.0,15.0,22.0,22.243243,25.121622,27.783784
257112,air_dea0655f96947922,2017-04-22,64,2016-01-02,0,Sat,0,-7,0,Dining bar,...,19.778918,19.499812,3.881836,32.0,30.0,55.0,54.0,22.243243,25.121622,27.783784
276170,air_eda179770dfa9f91,2017-04-22,21,2016-07-01,0,Sat,0,-7,0,Izakaya,...,19.778918,19.499812,3.881836,13.0,19.0,6.0,17.0,22.243243,25.121622,27.783784
280270,air_efef1e3daecce07e,2017-04-22,47,2016-01-05,0,Sat,0,-7,0,Other,...,19.778918,19.499812,3.881836,39.0,38.0,47.0,51.0,22.243243,25.121622,27.783784


#### Historical day-of-week mean of visitors


In [34]:
air_visit_df, future_df = add_historical_dow_mean(
    air_visit_df,
    VISITORS_COL,
    VISITORS_DOW_MEAN_COL,
    future_df
)
air_visit_df, future_df = add_historical_dow_mean(
    air_visit_df,
    VISITORS_NBR_COL,
    VISITORS_DOW_MEAN_NBR_COL,
    future_df
)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,...,visitors_nbrs_std_28,visitors_last_month,visitors_lag_1,visitors_lag_7,visitors_lag_28,visitors_nbrs_lag_1,visitors_nbrs_lag_7,visitors_nbrs_lag_28,visitors_dow_mean,visitors_dow_mean_nbrs
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1,Fri,0,-17,0,Italian/French,...,3.229918,NaN,NaN,NaN,NaN,19.949045,23.460123,21.920245,NaN,NaN
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1,Sat,0,-16,0,Italian/French,...,3.343359,NaN,35.0,NaN,NaN,24.935252,24.050000,22.565217,NaN,NaN
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1,Sun,0,-15,0,Italian/French,...,3.379464,NaN,9.0,NaN,NaN,23.669100,23.414634,21.830508,NaN,NaN
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1,Mon,0,-14,0,Italian/French,...,3.364158,NaN,0.0,NaN,NaN,20.609121,16.833333,15.160305,NaN,NaN
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1,Tue,0,-13,0,Italian/French,...,3.333447,NaN,20.0,NaN,NaN,15.793296,17.794872,16.903226,NaN,NaN


In [35]:
future_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,city,open_usually,days_from_last_visit,id,air_city_te,air_genre_te,total_reservations,total_reservations_nbrs,visitors_dow_mean,visitors_dow_mean_nbrs
0,air_00a91d42b08b08d9,2017-04-23,0,2016-07-01,0,Sun,0,-6,0,Italian/French,Tōkyō-to,0,1,air_00a91d42b08b08d9_2017-04-23,18.734807,20.680881,0.0,7.443182,0.048780,20.197084
1,air_00a91d42b08b08d9,2017-04-24,0,2016-07-01,0,Mon,0,-5,0,Italian/French,Tōkyō-to,1,2,air_00a91d42b08b08d9_2017-04-24,18.734807,20.680881,0.0,10.388350,18.707317,14.527403
2,air_00a91d42b08b08d9,2017-04-25,0,2016-07-01,0,Tue,0,-4,0,Italian/French,Tōkyō-to,1,0,air_00a91d42b08b08d9_2017-04-25,18.734807,20.680881,0.0,8.569038,22.902439,15.831236
3,air_00a91d42b08b08d9,2017-04-26,0,2016-07-01,0,Wed,0,-3,0,Italian/French,Tōkyō-to,1,0,air_00a91d42b08b08d9_2017-04-26,18.734807,20.680881,0.0,11.735354,27.024390,17.819776
4,air_00a91d42b08b08d9,2017-04-27,0,2016-07-01,0,Th,0,-2,0,Italian/French,Tōkyō-to,1,0,air_00a91d42b08b08d9_2017-04-27,18.734807,20.680881,0.0,9.649886,26.756098,17.512816


#### Rolling reserve/visitors difference


In [36]:
air_visit_df = add_reserves_difference(
    air_visit_df,
    VISITORS_COL,
    TOTAL_RES_COL,
    RES_VISITORS_DIFF_COL,
)
air_visit_df = add_reserves_difference(
    air_visit_df,
    VISITORS_NBR_COL,
    TOTAL_RES_NBR_COL,
    RES_VISITORS_DIFF_NBR_COL,
)

In [37]:
aggs = [("mean", {})]
air_visit_df = add_basic_stats(
    air_visit_df, RES_VISITORS_DIFF_COL, AIR_RESTAURANT_ID_COL, aggs
)
air_visit_df = add_neighbors_stats(
    air_visit_df, RES_VISITORS_DIFF_NBR_COL, CITY_COL, aggs, False
)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,...,visitors_dow_mean,visitors_dow_mean_nbrs,res_visitors_diff,res_visitors_diff_mean_7,res_visitors_diff_mean_14,res_visitors_diff_mean_28,nbr_res_visitors_diff,nbr_res_visitors_diff_mean_7,nbr_res_visitors_diff_mean_14,nbr_res_visitors_diff_mean_28
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1,Fri,0,-17,0,Italian/French,...,NaN,NaN,NaN,NaN,NaN,NaN,-11.430450,-11.965413,-12.257536,-12.295733
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1,Sat,0,-16,0,Italian/French,...,NaN,NaN,-35.0,NaN,NaN,NaN,-16.331325,-12.336299,-12.294435,-12.279419
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1,Sun,0,-15,0,Italian/French,...,NaN,NaN,-9.0,-35.000000,-35.000000,-35.000000,-15.364204,-12.861909,-12.355681,-12.375317
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1,Mon,0,-14,0,Italian/French,...,NaN,NaN,0.0,-22.000000,-22.000000,-22.000000,-13.766372,-12.792654,-12.266305,-12.384195
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1,Tue,0,-13,0,Italian/French,...,NaN,NaN,-20.0,-14.666667,-14.666667,-14.666667,-9.043419,-12.413971,-12.164135,-12.361298


#### NaN dropping

In [38]:
air_visit_df = drop_first_month(air_visit_df)
air_visit_df = air_visit_df.dropna()
air_visit_df = air_visit_df.sort_values(VISIT_DATE_COL)

In [39]:
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,golden_week_flg,air_genre_name,...,visitors_dow_mean,visitors_dow_mean_nbrs,res_visitors_diff,res_visitors_diff_mean_7,res_visitors_diff_mean_14,res_visitors_diff_mean_28,nbr_res_visitors_diff,nbr_res_visitors_diff_mean_7,nbr_res_visitors_diff_mean_14,nbr_res_visitors_diff_mean_28
244410,air_d0e8a085d8dc83aa,2016-02-01,10,2016-01-01,1,Mon,0,-10,0,Cafe/Sweets,...,15.75,18.633926,-18.0,-13.857143,-12.428571,-11.857143,-18.363894,-14.179130,-13.293905,-13.579609
253575,air_db4b38ebe7a7ceff,2016-02-01,14,2016-01-01,1,Mon,0,-10,0,Dining bar,...,12.00,15.657070,-10.0,-6.142857,-5.928571,-6.964286,-20.621858,-11.494365,-11.759436,-12.562822
241908,air_cfdeb326418194ff,2016-02-01,3,2016-01-01,1,Mon,0,-10,0,Bar/Cocktail,...,5.25,15.139980,0.0,-11.857143,-10.500000,-9.571429,-12.806373,-11.353651,-11.177742,-11.955113
61980,air_39dccf7df20b1c6a,2016-02-01,26,2016-01-01,1,Mon,0,-10,0,Izakaya,...,24.75,18.633926,-38.0,-18.714286,-21.928571,-23.571429,-18.363894,-14.179130,-13.293905,-13.579609
237464,air_cb7467aed805e7fe,2016-02-01,16,2016-01-01,1,Mon,0,-10,0,Izakaya,...,16.75,13.769616,-21.0,-30.142857,-27.428571,-25.821429,-14.251425,-7.888040,-8.603754,-10.349760


#### Saving

In [40]:
air_visit_df = air_visit_df.drop(columns=[OPEN_DATE_COL])
labels = air_visit_df[VISITORS_COL].copy()
features = air_visit_df.drop(columns=[VISITORS_COL])
future_df = future_df.drop(columns=[OPEN_DATE_COL, "id", VISITORS_COL])
future_df[TOTAL_RES_NBR_COL] = future_df[TOTAL_RES_NBR_COL].fillna(0)
features.to_csv(PROCESSED_DATA_DIR / "features.csv", index=False)
labels.to_csv(PROCESSED_DATA_DIR / "labels.csv", index=False)
future_df.to_csv(PROCESSED_DATA_DIR / "test_features.csv", index=False)

#### Feature correlation

In [41]:
corr = air_visit_df.drop(
    columns=[AIR_RESTAURANT_ID_COL, VISIT_DATE_COL, DAY_OF_WEEK_COL, AIR_GENRE_COL, CITY_COL]
).corr()
plot_corr_matrix(corr)

## Conclusion

### Added features.

- Operating schedule - regular working days, extracted from the regular gaps in the data frame.
- Days to/from the nearest holiday (negative is days until, positive is days after).
- Holiday indicator (currently included in date_info).
- Separate indicators for Golden Week dates.

- Number of visitors on this day last month for this restaurant.

- Indicator of whether the restaurant has been open within the last 6 months.
- Days since last recorded visit for air_visit dataframe.

- Rolling mean/median/std of visitors over the past week, month - for this restaurant/for neighbors.
- Historical day-of-week mean up to (but not including) the current day - for this restaurant/for neighbors.
- Rolling reserve/visitors difference over the past 7 / 28 days - for this restaurant/for neighbors.
- Lag 1, 7, 28 of the number of visitors - for this restaurant/for neighbors.

- Smoothed target encoding of visitors by air_area_name.
- Smoothed target encoding of visitors by air_genre_name.

- Total reserved visitors (from air_reserve and hpg_reserve) on this day - for this restaurant/for neighbors.

### Features planned for addition.

- Days since last recorded visit for sample_submission.

### Possible features.

- Daily temperature - try to get the weather forecast.
- Precipitation probability.
- Hpg genre.

Decomposition features:
- Trend.
- Trend difference for last month.
- Seasonal.
- Residual mean for last month.